# Nucleotide Transformer × DeepChem — Quick Demo
> **GSoC 2026** | Single Cell and DNA Foundation Models

This notebook shows the full pipeline in under 5 minutes.

In [ ]:
import numpy as np
import deepchem as dc
from deepchem.models.torch_models.nucleotide_transformer import NucleotideTransformerModel, NUCLEOTIDE_TRANSFORMER_MODELS
from deepchem.feat.sequence_featurizers.dna_tokenizer_featurizer import DNATokenizerFeaturizer, KMerDNAFeaturizer
print('Available models:', list(NUCLEOTIDE_TRANSFORMER_MODELS.keys()))

## 1. Featurize DNA sequences

In [ ]:
seqs = [
    'ATCGATCGATCGATCGATCGATCG',
    'GCTAGCTAGCTAGCTAGCTAGCTA',
    'TTTTAAAACCCCGGGGTTTTAAAA',
    'ACGTACGTACGTACGTACGTACGT',
]

feat = DNATokenizerFeaturizer(
    tokenizer_path='InstaDeepAI/nucleotide-transformer-v2-100m-multi-species',
    max_length=128)
token_ids = feat.featurize(seqs)
print('Token IDs shape:', token_ids.shape)

kmer_feat = KMerDNAFeaturizer(k=6)
kmer_vecs = kmer_feat.featurize(seqs)
print('6-mer vector shape:', kmer_vecs.shape)

## 2. Build a DeepChem dataset from raw DNA strings

In [ ]:
labels = np.array([[1],[0],[1],[0]], dtype=np.float32)
dataset = dc.data.NumpyDataset(
    X=np.array(seqs, dtype=object),
    y=labels,
    ids=seqs)
print(f'Dataset: {len(dataset)} molecules, {dataset.y.shape[1]} task(s)')

## 3. Scaffold split

In [ ]:
splitter = dc.splits.RandomSplitter()
train, valid, test = splitter.train_valid_test_split(dataset)
print(f'Train={len(train)} Valid={len(valid)} Test={len(test)}')

## 4. Fine-tune NucleotideTransformer for classification

In [ ]:
model = NucleotideTransformerModel(
    n_tasks=1,
    mode='classification',
    model_path='v2-100m-multi-species',
    max_seq_length=128,
    batch_size=2,
    learning_rate=2e-5,
)
print('Trainable params:', sum(p.numel() for p in model.model.parameters() if p.requires_grad))
loss = model.fit(train, nb_epoch=3)
print(f'Training loss: {loss:.4f}')

## 5. Evaluate

In [ ]:
metric = dc.metrics.Metric(dc.metrics.roc_auc_score)
scores = model.evaluate(test, [metric])
print('Test ROC-AUC:', scores)

## 6. Extract embeddings for downstream analysis

In [ ]:
embeddings = model.get_embeddings(seqs, pooling='mean')
print('Embedding matrix shape:', embeddings.shape)

from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
pca  = PCA(n_components=2)
comp = pca.fit_transform(embeddings)
fig, ax = plt.subplots(figsize=(5,4))
ax.scatter(comp[:,0], comp[:,1], c=labels[:,0], cmap='coolwarm', s=80)
ax.set_xlabel('PC 1'); ax.set_ylabel('PC 2')
ax.set_title('NT embeddings — PCA projection')
plt.tight_layout()
plt.savefig('../docs/embedding_pca.png', dpi=150)
plt.show()
print('PCA saved to docs/embedding_pca.png')